# 08 - Regularización y Evaluación de Modelos

**AI sin humo** - Notas personales para entender deep learning desde cero.

En el notebook anterior vimos todas las técnicas para hacer que el entrenamiento **funcione**: SGD, Adam, BatchNorm, residual connections, etc. Todo genial. El modelo aprende y la loss baja. Pero hay un problema gigante que todavía no atacamos: **¿qué pasa cuando el modelo aprende DEMASIADO bien los datos de entrenamiento?**

Este notebook es sobre **regularización**: cómo evitar que el modelo memorice los datos en vez de aprender patrones generales. Si la loss de entrenamiento baja a cero pero el modelo falla con datos nuevos, tenemos un problema serio. Acá vemos por qué pasa y todas las herramientas para combatirlo.

---

## Contenido

1. [Overfitting y generalización](#overfitting)
2. [Regularización explícita](#reg-explicita)
3. [L2 regularization (weight decay)](#l2)
4. [Regularización implícita](#reg-implicita)
5. [Early stopping](#early-stopping)
6. [Ensembling](#ensembling)
7. [Dropout](#dropout)
8. [Agregar ruido](#ruido)
9. [Transfer learning](#transfer-learning)
10. [Data augmentation](#data-augmentation)
11. [Resumen](#resumen)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

---

<a id='overfitting'></a>
## 1. Overfitting y generalización

### El objetivo real del machine learning

Vamos a ser claros sobre algo fundamental: el objetivo de entrenar un modelo **no es** que ande bien con los datos de entrenamiento. El objetivo es que ande bien con **datos que nunca vio**. A esto le llamamos **generalización**.

Pensalo así: si le damos a un modelo 1000 fotos de gatos y perros para que aprenda, no nos importa que clasifique bien esas 1000 fotos (ya sabemos qué son). Nos importa que clasifique bien la foto **número 1001**, que nunca vio.

### Training error vs test error

Para medir esto, separamos los datos en dos conjuntos:

- **Training set**: los datos que usa el modelo para aprender (ajustar parámetros)
- **Test set**: datos que el modelo **nunca ve** durante el entrenamiento, solo para evaluar

$$\text{Training error} = \frac{1}{N_{train}} \sum_{i \in \text{train}} L(f(x_i; \theta), y_i)$$

$$\text{Test error} = \frac{1}{N_{test}} \sum_{i \in \text{test}} L(f(x_i; \theta), y_i)$$

La diferencia entre estos dos nos dice todo:

| Situación | Training error | Test error | Diagnóstico |
|:----------|:---------------|:-----------|:------------|
| Training error alto, test error alto | Alto | Alto | **Underfitting** — el modelo es demasiado simple |
| Training error bajo, test error bajo | Bajo | Bajo | **Bien** — el modelo generaliza |
| Training error bajo, test error alto | Bajo | Alto | **Overfitting** — memoriza en vez de aprender |

### ¿Qué es overfitting exactamente?

Overfitting es cuando el modelo se ajusta no solo a los **patrones reales** de los datos, sino también al **ruido** y a las **peculiaridades** del training set. El modelo "memoriza" los datos de entrenamiento en vez de aprender la estructura subyacente.

Una red neuronal muy expresiva (muchas capas, muchas neuronas) tiene la **capacidad** de memorizar cualquier dataset. Si le das suficiente capacidad, puede asignar a cada punto de entrenamiento exactamente el output correcto, pero lo hace de una forma tan complicada y retorcida que falla miserablemente con datos nuevos.

### El bias-variance tradeoff

Esta es una de las ideas más importantes de todo machine learning. El error de un modelo se puede descomponer en tres componentes:

$$\text{Error total} = \text{Bias}^2 + \text{Variance} + \text{Ruido irreducible}$$

- **Bias**: error por simplificar demasiado el problema. Un modelo con bias alto "no entiende" la estructura de los datos. Ejemplo: usar una línea recta para datos que son curvos → underfitting.

- **Variance**: error por ser demasiado sensible a los datos de entrenamiento. Un modelo con varianza alta cambia drásticamente si cambiás un par de datos del training set. Ejemplo: un polinomio de grado 100 para 10 datos → overfitting.

- **Ruido irreducible**: el ruido inherente a los datos. No importa qué tan perfecto sea tu modelo, no podés predecir mejor que esto.

El tradeoff es que al hacer el modelo **más complejo**:
- El bias **baja** (puede capturar patrones más complicados)
- La varianza **sube** (se ajusta más a los datos específicos de entrenamiento)

El punto óptimo es donde la suma es mínima. Y todas las técnicas de regularización que vamos a ver buscan **bajar la varianza sin subir demasiado el bias**.

In [ ]:
# Demo: overfitting with polynomials of increasing degree
np.random.seed(42)

# True function: simple sine curve
def true_function(x):
    return np.sin(2 * x)

# Generate training and test data
n_train = 15
n_test = 200
noise_std = 0.3

x_train = np.sort(np.random.uniform(-3, 3, n_train))
y_train = true_function(x_train) + noise_std * np.random.randn(n_train)

x_test = np.linspace(-3, 3, n_test)
y_test = true_function(x_test) + noise_std * np.random.randn(n_test)
x_plot = np.linspace(-3.2, 3.2, 500)

# Fit polynomials of different degrees
degrees = [1, 3, 5, 14]
titles = ['Grado 1 (underfitting)', 'Grado 3 (bien)', 'Grado 5 (razonable)', f'Grado {n_train-1} (overfitting)']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

train_errors = []
test_errors = []

for ax, degree, title in zip(axes, degrees, titles):
    # Fit polynomial
    coeffs = np.polyfit(x_train, y_train, degree)
    poly = np.poly1d(coeffs)
    
    # Compute errors
    train_pred = poly(x_train)
    test_pred = poly(x_test)
    train_mse = np.mean((y_train - train_pred) ** 2)
    test_mse = np.mean((y_test - test_pred) ** 2)
    train_errors.append(train_mse)
    test_errors.append(test_mse)
    
    # Plot
    ax.plot(x_plot, true_function(x_plot), 'k--', alpha=0.4, linewidth=1, label='f(x) real')
    ax.scatter(x_train, y_train, c='steelblue', s=40, zorder=5, label='Train')
    y_pred_plot = poly(x_plot)
    y_pred_plot = np.clip(y_pred_plot, -5, 5)  # clip for visualization
    ax.plot(x_plot, y_pred_plot, 'r-', linewidth=2, label=f'Polinomio grado {degree}')
    ax.set_title(f'{title}\ntrain={train_mse:.3f}, test={test_mse:.3f}', fontsize=10)
    ax.set_ylim(-3, 3)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Bias-Variance Tradeoff: complejidad del modelo', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

print("Grado 1:  demasiado simple → no captura la curva (BIAS alto)")
print("Grado 3:  justo → captura la forma sin ajustar al ruido")
print("Grado 5:  empieza a temblar un poco")
print(f"Grado {n_train-1}: pasa por TODOS los puntos → train error ≈ 0 pero test error altísimo (VARIANZA alta)")

In [ ]:
# Train vs test error as a function of model complexity
all_degrees = range(1, 15)
all_train_errors = []
all_test_errors = []

for d in all_degrees:
    coeffs = np.polyfit(x_train, y_train, d)
    poly = np.poly1d(coeffs)
    all_train_errors.append(np.mean((y_train - poly(x_train)) ** 2))
    all_test_errors.append(np.mean((y_test - poly(x_test)) ** 2))

# Clip test errors for visualization
all_test_errors_clipped = np.clip(all_test_errors, 0, 3)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(all_degrees), all_train_errors, 'bo-', linewidth=2, markersize=6, label='Training error')
ax.plot(list(all_degrees), all_test_errors_clipped, 'ro-', linewidth=2, markersize=6, label='Test error')

# Mark the sweet spot
best_degree = list(all_degrees)[np.argmin(all_test_errors)]
ax.axvline(x=best_degree, color='green', linestyle='--', alpha=0.5, label=f'Mejor: grado {best_degree}')

# Annotations
ax.annotate('UNDERFITTING\n(bias alto)', xy=(1.5, 0.6), fontsize=11, color='purple',
            ha='center', fontweight='bold')
ax.annotate('OVERFITTING\n(varianza alta)', xy=(12, 1.5), fontsize=11, color='purple',
            ha='center', fontweight='bold')
ax.annotate('← sweet spot →', xy=(best_degree, 0.05), fontsize=10, color='green',
            ha='center', fontweight='bold')

ax.set_xlabel('Complejidad del modelo (grado del polinomio)', fontsize=11)
ax.set_ylabel('Error (MSE)', fontsize=11)
ax.set_title('La U clásica: training error SIEMPRE baja, test error sube con overfitting', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 3)
plt.tight_layout()
plt.show()

print("Este gráfico es LA IMAGEN más importante de machine learning.")
print("Training error SIEMPRE baja con más complejidad (puede memorizar más).")
print("Test error baja al principio (el modelo mejora) y SUBE después (overfitting).")
print("\nTodo lo que vamos a ver en este notebook busca mover esa curva roja para abajo.")

In [ ]:
# Neural network overfitting demo: small vs large network
np.random.seed(42)

# Generate a simple 1D dataset
n_train = 30
x_train_nn = np.random.uniform(-2, 2, n_train).reshape(-1, 1)
y_train_nn = np.sin(3 * x_train_nn) + 0.3 * np.random.randn(n_train, 1)
x_test_nn = np.linspace(-2.5, 2.5, 300).reshape(-1, 1)

# Simple neural network (from scratch, just for this demo)
def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

class SimpleNet:
    """2-layer neural network for 1D regression."""
    def __init__(self, hidden_size, seed=42):
        np.random.seed(seed)
        self.W1 = np.random.randn(1, hidden_size) * np.sqrt(2.0 / 1)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, 1) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, 1))
    
    def forward(self, x):
        self.z1 = x @ self.W1 + self.b1
        self.h1 = relu(self.z1)
        self.out = self.h1 @ self.W2 + self.b2
        return self.out
    
    def backward(self, x, y, lr=0.01):
        n = x.shape[0]
        # dL/d_out
        d_out = 2 * (self.out - y) / n
        # Layer 2 grads
        dW2 = self.h1.T @ d_out
        db2 = d_out.sum(axis=0, keepdims=True)
        # Layer 1 grads
        d_h1 = d_out @ self.W2.T
        d_z1 = d_h1 * relu_grad(self.z1)
        dW1 = x.T @ d_z1
        db1 = d_z1.sum(axis=0, keepdims=True)
        # Update
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
    
    def loss(self, x, y):
        pred = self.forward(x)
        return np.mean((pred - y) ** 2)

# Train small network (5 hidden units) and large network (200 hidden units)
small_net = SimpleNet(hidden_size=5, seed=42)
large_net = SimpleNet(hidden_size=200, seed=42)

epochs = 3000
lr = 0.01
small_train_losses = []
large_train_losses = []

for epoch in range(epochs):
    # Small net
    small_net.forward(x_train_nn)
    small_net.backward(x_train_nn, y_train_nn, lr=lr)
    small_train_losses.append(small_net.loss(x_train_nn, y_train_nn))
    
    # Large net
    large_net.forward(x_train_nn)
    large_net.backward(x_train_nn, y_train_nn, lr=lr)
    large_train_losses.append(large_net.loss(x_train_nn, y_train_nn))

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Small network predictions
ax = axes[0]
y_pred_small = small_net.forward(x_test_nn)
ax.scatter(x_train_nn, y_train_nn, c='steelblue', s=30, zorder=5, label='Train data')
ax.plot(x_test_nn, np.sin(3 * x_test_nn), 'k--', alpha=0.4, label='f(x) real')
ax.plot(x_test_nn, y_pred_small, 'r-', linewidth=2, label='Red (5 neuronas)')
ax.set_title(f'Red chica (5 neuronas)\n→ Underfits un poco', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(-2.5, 2.5)
ax.grid(True, alpha=0.3)

# Large network predictions
ax = axes[1]
y_pred_large = large_net.forward(x_test_nn)
ax.scatter(x_train_nn, y_train_nn, c='steelblue', s=30, zorder=5, label='Train data')
ax.plot(x_test_nn, np.sin(3 * x_test_nn), 'k--', alpha=0.4, label='f(x) real')
ax.plot(x_test_nn, np.clip(y_pred_large, -3, 3), 'r-', linewidth=2, label='Red (200 neuronas)')
ax.set_title(f'Red grande (200 neuronas)\n→ Overfits (se retuerce)', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(-2.5, 2.5)
ax.grid(True, alpha=0.3)

# Training losses
ax = axes[2]
ax.plot(small_train_losses, 'b-', alpha=0.7, label='Red chica (5)')
ax.plot(large_train_losses, 'r-', alpha=0.7, label='Red grande (200)')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Train Loss (MSE)', fontsize=11)
ax.set_title('La red grande tiene menor train loss\npero eso NO significa que sea mejor', fontsize=11)
ax.legend(fontsize=10)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Red chica (5 neuronas):   {sum(p.size for p in [small_net.W1, small_net.b1, small_net.W2, small_net.b2]):,} parámetros")
print(f"Red grande (200 neuronas): {sum(p.size for p in [large_net.W1, large_net.b1, large_net.W2, large_net.b2]):,} parámetros")
print(f"\nLa red grande tiene más parámetros que datos de entrenamiento ({n_train}).")
print(f"Puede MEMORIZAR cada punto. Eso es overfitting.")

Ese es el problema central: la red grande tiene un training loss más bajo pero anda peor donde importa (datos nuevos). Tiene la capacidad de memorizar cada punto de entrenamiento y lo hace, generando una función super irregular que pasa exactamente por cada dato pero no captura el patrón real.

### ¿Y entonces qué hacemos?

Tenemos dos opciones:

1. **Restringir el modelo**: usar menos neuronas, menos capas. Pero esto limita qué tan complejo puede ser el patrón que captura. En la práctica, **no queremos hacer esto**. Queremos redes grandes y poderosas.

2. **Regularizar**: mantener la red grande pero agregar restricciones que la guíen hacia soluciones más simples. Esto es lo que vamos a ver ahora.

La idea es: dame un modelo con **capacidad suficiente** para capturar cualquier patrón, pero agregale incentivos para que prefiera soluciones simples y suaves. Si el patrón real es complejo, el modelo tiene la capacidad de capturarlo. Si es simple, la regularización lo empuja hacia la solución simple.

---

<a id='reg-explicita'></a>
## 2. Regularización explícita

La forma más directa de regularizar es modificar la **loss function**. En vez de solo minimizar el error, le agregamos un **término de penalización** que castiga ciertos comportamientos indeseables de los parámetros:

$$\mathcal{L}_{\text{reg}} = \mathcal{L}_{\text{data}}(\theta) + \lambda \cdot g(\theta)$$

Donde:
- $\mathcal{L}_{\text{data}}(\theta)$ es la loss original (MSE, cross-entropy, etc.)
- $g(\theta)$ es una función que **penaliza** ciertas propiedades de los parámetros
- $\lambda$ controla qué tan fuerte es la penalización (hiperparámetro)

### Es como un prior bayesiano

Esto tiene una interpretación muy elegante desde la perspectiva bayesiana. El término $\mathcal{L}_{\text{data}}$ viene de los datos (el likelihood). El término $\lambda \cdot g(\theta)$ viene de nuestro **conocimiento previo** sobre cómo deberían ser los parámetros (el prior).

Es como decirle al modelo: "aprendé de los datos, pero también tené en cuenta que yo creo que los parámetros deberían ser [chicos / dispersos / suaves / etc.]". Si los datos son abundantes y claros, el modelo los sigue. Si los datos son ambiguos o escasos, el prior domina y guía al modelo hacia soluciones razonables.

$$\underbrace{\mathcal{L}_{\text{reg}}}_{\text{MAP estimation}} = \underbrace{\mathcal{L}_{\text{data}}}_{-\log P(\text{data}|\theta)} + \underbrace{\lambda \cdot g(\theta)}_{-\log P(\theta)}$$

![Loss + Regularización: el regularizador empuja la solución hacia el centro (parámetros chicos)](../ai_notas/AI%20notas/image%2048.png)

La imagen de arriba muestra exactamente lo que pasa:
- **(a) Loss sola**: la superficie tiene muchos mínimos, algunos en regiones de parámetros grandes.
- **(b) Regularización sola**: el regularizador penaliza parámetros grandes, creando un "pozo" centrado en cero.
- **(c) Loss + regularización**: al combinarlos, los mínimos se desplazan hacia la zona de parámetros más chicos. Los mínimos "malos" (parámetros grandes) desaparecen.

### ¿Qué puede penalizar $g(\theta)$?

Las funciones de penalización más comunes son:

| Regularización | $g(\theta)$ | Efecto |
|:---------------|:------------|:-------|
| **L2 (weight decay)** | $\sum_i \theta_i^2$ | Parámetros chicos, funciones suaves |
| **L1 (Lasso)** | $\sum_i |\theta_i|$ | Parámetros dispersos (muchos exactamente en 0) |
| **Elastic Net** | $\alpha \sum_i |\theta_i| + (1-\alpha) \sum_i \theta_i^2$ | Combinación de L1 y L2 |

Veamos la más importante en deep learning: L2.

---

<a id='l2'></a>
## 3. L2 regularization (weight decay)

La regularización L2 penaliza la **suma de cuadrados** de todos los parámetros:

$$\mathcal{L}_{\text{reg}} = \mathcal{L}_{\text{data}}(\theta) + \frac{\lambda}{2} \sum_i \theta_i^2$$

(El $\frac{1}{2}$ es para que la derivada sea más limpia.)

### ¿Qué efecto tiene?

**1. Parámetros más chicos → función más suave**

Cuando los weights son chicos, la función que computa la red es más **suave** (cambia gradualmente). Cuando los weights son grandes, la función puede oscilar salvajemente. L2 empuja todos los weights hacia cero, haciendo que la red prefiera funciones simples y suaves.

**2. Más conservador donde no hay datos**

En regiones donde no hay datos de entrenamiento, la red no tiene información para decidir qué hacer. Sin regularización, puede hacer cualquier cosa (incluyendo cosas locas). Con L2, la red se "relaja" a una función suave en esas regiones porque los parámetros son chicos.

**3. Reduce la dependencia de pocos features**

Si un feature tiene un weight gigante y otro tiene un weight chiquito, el modelo depende casi completamente del primer feature. L2 distribuye los weights más uniformemente, haciendo que el modelo use todos los features de forma más balanceada.

### El gradiente con L2

El gradiente de la loss regularizada es:

$$\nabla_{\theta} \mathcal{L}_{\text{reg}} = \nabla_{\theta} \mathcal{L}_{\text{data}} + \lambda \theta$$

Y la actualización de SGD:

$$\theta \leftarrow \theta - \eta (\nabla_{\theta} \mathcal{L}_{\text{data}} + \lambda \theta) = (1 - \eta \lambda) \theta - \eta \nabla_{\theta} \mathcal{L}_{\text{data}}$$

Fijate: en cada paso, el parámetro se multiplica por $(1 - \eta \lambda)$, que es un número menor que 1. Esto **"decae" los weights** en cada paso, de ahí el nombre **weight decay**.

### Weight decay vs L2 regularization

En SGD puro, weight decay y L2 regularization son equivalentes. Pero en Adam, **no lo son**. En Adam, el gradiente se normaliza por la varianza, así que el término $\lambda \theta$ también se normaliza, lo cual cambia el efecto. Por eso existe **AdamW**, que implementa weight decay directamente (sin pasar por el normalizador) y es la versión que todo el mundo usa hoy.

In [ ]:
# Demo: L2 regularization taming overfitting
np.random.seed(42)

# Reuse our SimpleNet but add L2 regularization
class SimpleNetL2(SimpleNet):
    """2-layer neural network with L2 regularization."""
    def backward(self, x, y, lr=0.01, weight_decay=0.0):
        n = x.shape[0]
        d_out = 2 * (self.out - y) / n
        # Layer 2 grads (+ L2 penalty gradient)
        dW2 = self.h1.T @ d_out + weight_decay * self.W2
        db2 = d_out.sum(axis=0, keepdims=True)
        # Layer 1 grads
        d_h1 = d_out @ self.W2.T
        d_z1 = d_h1 * relu_grad(self.z1)
        dW1 = x.T @ d_z1 + weight_decay * self.W1
        db1 = d_z1.sum(axis=0, keepdims=True)
        # Update
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
    
    def total_weight_norm(self):
        return np.sum(self.W1**2) + np.sum(self.W2**2)

# Train large network (200 hidden) with different λ values
lambdas = [0.0, 0.001, 0.01, 0.1]
labels = ['λ=0 (sin reg)', 'λ=0.001', 'λ=0.01', 'λ=0.1']
colors = ['red', 'orange', 'blue', 'green']

nets = []
all_train_losses_l2 = []
all_weight_norms = []

epochs = 3000
for lam in lambdas:
    net = SimpleNetL2(hidden_size=200, seed=42)
    train_losses = []
    weight_norms = []
    for epoch in range(epochs):
        net.forward(x_train_nn)
        net.backward(x_train_nn, y_train_nn, lr=0.01, weight_decay=lam)
        train_losses.append(net.loss(x_train_nn, y_train_nn))
        weight_norms.append(net.total_weight_norm())
    nets.append(net)
    all_train_losses_l2.append(train_losses)
    all_weight_norms.append(weight_norms)

# Plot predictions
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, net, label, color, lam in zip(axes, nets, labels, colors, lambdas):
    y_pred = net.forward(x_test_nn)
    ax.scatter(x_train_nn, y_train_nn, c='steelblue', s=25, zorder=5, label='Train')
    ax.plot(x_test_nn, np.sin(3 * x_test_nn), 'k--', alpha=0.4, label='f(x) real')
    ax.plot(x_test_nn, np.clip(y_pred, -3, 3), color=color, linewidth=2, label=label)
    ax.set_title(f'{label}\n||W||² = {net.total_weight_norm():.1f}', fontsize=10)
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Efecto de L2 regularization: misma red grande (200 neuronas)', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

# Plot weight norms over training
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for losses, label, color in zip(all_train_losses_l2, labels, colors):
    ax.plot(losses, color=color, alpha=0.7, label=label)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Train Loss', fontsize=11)
ax.set_title('Training loss (más regularización → loss más alta)', fontsize=11)
ax.legend(fontsize=9)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

ax = axes[1]
for wnorms, label, color in zip(all_weight_norms, labels, colors):
    ax.plot(wnorms, color=color, alpha=0.7, label=label)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('||W||² (norma de weights)', fontsize=11)
ax.set_title('Norma de weights (L2 los mantiene chicos)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Sin regularización: la red se retuerce para pasar por cada punto → overfitting")
print("Con λ=0.01: la red suaviza la función, captura el patrón sin ajustar al ruido")
print("Con λ=0.1:  demasiada regularización → la red es muy rígida → empieza a underfittear")
print("\nFijate que el training loss SUBE con regularización. Eso está bien.")
print("El objetivo no es minimizar el training loss, es generalizar.")

Fijate algo clave: con L2 regularization, la red grande con 200 neuronas se comporta **casi igual de bien** que la red chica con 5 neuronas, pero mantiene la **capacidad** de capturar patrones más complejos si los datos lo justifican. Esa es la magia de la regularización: no limitás la capacidad del modelo, solo lo guiás hacia soluciones más simples.

---

<a id='reg-implicita'></a>
## 4. Regularización implícita

No toda la regularización viene de modificar la loss function. Algunas técnicas que usamos por otras razones (eficiencia, estabilidad) resultan tener un **efecto regularizador** como bonus. A esto le llamamos regularización **implícita**.

### SGD como regularizador

Esto es una de las observaciones más sorprendentes de deep learning moderno. El hecho de usar SGD (con minibatches aleatorios) en vez de gradient descent completo tiene un efecto regularizador importante.

¿Por qué? Recordá que SGD usa un gradiente **ruidoso** calculado sobre un minibatch. Este ruido tiene un efecto particular: tiende a expulsar al modelo de mínimos **estrechos** (sharp minima) y mantenerlo en mínimos **anchos** (flat minima).

- **Mínimos estrechos**: la loss es baja en un punto muy específico del espacio de parámetros, pero sube rápido si te movés un poco. Estos mínimos **no generalizan bien** porque cualquier pequeña perturbación en los datos cambia el mínimo.
- **Mínimos anchos**: la loss es baja en toda una **región** del espacio de parámetros. Estos generalizan mejor porque son robustos a perturbaciones.

SGD, con su ruido, no puede quedarse en un mínimo estrecho — el ruido lo saca. Pero sí puede quedarse en uno ancho, porque el ruido no es suficiente para salir de la región.

### Mini-batches más chicos generalizan mejor

Acá viene algo contra-intuitivo: **batches más chicos dan más ruido, y eso generaliza mejor**. Varios estudios empíricos han mostrado que:

- Entrenar con batch size 32 generaliza mejor que entrenar con batch size 1024
- Batch sizes muy grandes pueden overfittear más, a pesar de que la optimización es más estable

Esto se explica exactamente por el argumento de los mínimos anchos vs estrechos: más ruido → el optimizador favorece mínimos anchos → mejor generalización.

### Otras formas de regularización implícita

- **Gradient descent con learning rate finito**: el hecho de dar pasos discretos (en vez de seguir el gradiente de forma continua) introduce un sesgo que favorece ciertas soluciones.
- **Arquitectura de la red**: la estructura misma de la red (convoluciones, atención, etc.) actúa como un prior implícito sobre qué tipo de funciones puede aprender.
- **Número finito de iteraciones**: no entrenar hasta converger completamente es regularización implícita (relacionado con early stopping, que vemos a continuación).

In [ ]:
# Demo: effect of batch size on generalization
np.random.seed(42)

# Generate more data for this demo
n_data = 200
x_data = np.random.uniform(-3, 3, (n_data, 1))
y_data = np.sin(2 * x_data) + 0.3 * np.random.randn(n_data, 1)

# Split train/test
x_tr, y_tr = x_data[:150], y_data[:150]
x_te, y_te = x_data[150:], y_data[150:]

def train_with_batch_size(batch_size, epochs=500, seed=42):
    """Train a network with a given batch size, track train and test loss."""
    net = SimpleNet(hidden_size=100, seed=seed)
    train_losses = []
    test_losses = []
    n = len(x_tr)
    
    for epoch in range(epochs):
        # Shuffle
        perm = np.random.permutation(n)
        x_shuffled = x_tr[perm]
        y_shuffled = y_tr[perm]
        
        # Mini-batch training
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            x_batch = x_shuffled[start:end]
            y_batch = y_shuffled[start:end]
            net.forward(x_batch)
            net.backward(x_batch, y_batch, lr=0.01)
        
        # Track losses (on full sets)
        train_losses.append(net.loss(x_tr, y_tr))
        test_losses.append(net.loss(x_te, y_te))
    
    return net, train_losses, test_losses

batch_sizes = [8, 32, 150]  # small, medium, full-batch
bs_labels = ['Batch=8 (ruidoso)', 'Batch=32 (típico)', 'Batch=150 (full-batch)']
bs_colors = ['green', 'blue', 'red']

results = []
for bs in batch_sizes:
    net, train_l, test_l = train_with_batch_size(bs, epochs=300)
    results.append((net, train_l, test_l))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for (net, train_l, _), label, color in zip(results, bs_labels, bs_colors):
    ax.plot(train_l, color=color, alpha=0.7, label=label)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Train Loss', fontsize=11)
ax.set_title('Training loss por batch size', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
for (net, _, test_l), label, color in zip(results, bs_labels, bs_colors):
    ax.plot(test_l, color=color, alpha=0.7, label=label)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Test Loss', fontsize=11)
ax.set_title('Test loss por batch size (generalización)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Batch size chico → más ruido → regularización implícita → mejor test loss")
print("Full-batch → sin ruido → puede overfittear más")
print(f"\nTest loss final batch=8:   {results[0][2][-1]:.4f}")
print(f"Test loss final batch=32:  {results[1][2][-1]:.4f}")
print(f"Test loss final batch=150: {results[2][2][-1]:.4f}")

---

<a id='early-stopping'></a>
## 5. Early stopping

Early stopping es la técnica de regularización más simple y pragmática: **parar el entrenamiento antes de que el modelo converja completamente**.

### La idea

Durante el entrenamiento, el training loss baja continuamente (eso es lo que el optimizador hace). Pero el test/validation loss típicamente hace esto:

1. **Baja** al principio (el modelo está aprendiendo patrones útiles)
2. **Llega a un mínimo** (punto óptimo de generalización)
3. **Sube** después (el modelo empieza a memorizar ruido → overfitting)

Early stopping dice: guardá el modelo en el punto 2, antes de que llegue al punto 3.

### En la práctica

1. Separás un **validation set** (distinto del test set)
2. En cada epoch, evaluás la loss en el validation set
3. Guardás el modelo cada vez que el validation loss mejora
4. Si el validation loss no mejora por $p$ epochs ("patience"), parás
5. Usás el último modelo guardado (no el del final del entrenamiento)

```python
best_val_loss = float('inf')
patience_counter = 0
patience = 20

for epoch in range(max_epochs):
    train_one_epoch(model)
    val_loss = evaluate(model, val_set)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_model(model)         # guardar el mejor modelo
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            break                 # parar!

model = load_model()              # usar el mejor, no el último
```

### ¿Por qué funciona? Conexión con L2

Hay un resultado teórico muy lindo: si los parámetros empiezan **cerca de cero** (que es lo que pasa con inicialización estándar), early stopping es aproximadamente equivalente a L2 regularization.

La intuición es: si empezás con parámetros chiquitos y parás temprano, los parámetros nunca llegan a ser muy grandes. Cuanto antes parás, más chiquitos se quedan → más regularización. Cuanto más tarde parás, más grandes pueden crecer → menos regularización.

Entonces, el número de epochs de entrenamiento actúa como una especie de hiperparámetro de regularización inverso: **menos epochs = más regularización**.

### Ventajas de early stopping

- **Gratis**: no necesitás cambiar la loss ni la arquitectura
- **Automática**: no hay un λ que tunear (el patience es fácil de elegir)
- **Siempre funciona**: es la primera línea de defensa contra overfitting

In [ ]:
# Demo: early stopping
np.random.seed(42)

# Train a large network and track train/val/test loss
n_total = 200
x_all = np.random.uniform(-3, 3, (n_total, 1))
y_all = np.sin(2 * x_all) + 0.3 * np.random.randn(n_total, 1)

# Split: 60% train, 20% val, 20% test
x_tr_es = x_all[:120]
y_tr_es = y_all[:120]
x_val = x_all[120:160]
y_val = y_all[120:160]
x_te_es = x_all[160:]
y_te_es = y_all[160:]

# Train large network for many epochs
net_es = SimpleNet(hidden_size=100, seed=42)
train_losses_es = []
val_losses_es = []
test_losses_es = []

best_val_loss = float('inf')
best_epoch = 0
best_weights = None

for epoch in range(2000):
    # Train step
    net_es.forward(x_tr_es)
    net_es.backward(x_tr_es, y_tr_es, lr=0.01)
    
    # Track losses
    t_loss = net_es.loss(x_tr_es, y_tr_es)
    v_loss = net_es.loss(x_val, y_val)
    te_loss = net_es.loss(x_te_es, y_te_es)
    train_losses_es.append(t_loss)
    val_losses_es.append(v_loss)
    test_losses_es.append(te_loss)
    
    # Save best model
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_epoch = epoch
        best_weights = (net_es.W1.copy(), net_es.b1.copy(),
                        net_es.W2.copy(), net_es.b2.copy())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(train_losses_es, 'b-', alpha=0.7, label='Train loss')
ax.plot(val_losses_es, 'r-', alpha=0.7, label='Validation loss')
ax.plot(test_losses_es, 'g--', alpha=0.5, label='Test loss')
ax.axvline(x=best_epoch, color='black', linestyle=':', linewidth=2, label=f'Best model (epoch {best_epoch})')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss (MSE)', fontsize=11)
ax.set_title('Early stopping: parar en el mínimo de validation', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, max(val_losses_es[0], train_losses_es[0]) * 1.1)

# Show predictions at different epochs
ax = axes[1]
x_plot_es = np.linspace(-3.5, 3.5, 300).reshape(-1, 1)

# Best model predictions
net_best = SimpleNet(hidden_size=100, seed=42)
net_best.W1, net_best.b1, net_best.W2, net_best.b2 = best_weights
y_best = net_best.forward(x_plot_es)

# Final (overfit) model predictions
y_final = net_es.forward(x_plot_es)

ax.scatter(x_tr_es, y_tr_es, c='steelblue', s=20, alpha=0.5, label='Train')
ax.scatter(x_val, y_val, c='orange', s=30, zorder=5, label='Validation')
ax.plot(x_plot_es, np.sin(2 * x_plot_es), 'k--', alpha=0.4, label='f(x) real')
ax.plot(x_plot_es, np.clip(y_best, -3, 3), 'g-', linewidth=2, label=f'Early stop (epoch {best_epoch})')
ax.plot(x_plot_es, np.clip(y_final, -3, 3), 'r-', linewidth=1.5, alpha=0.6, label='Final (epoch 2000)')
ax.set_title('Predicciones: early stop vs entrenamiento completo', fontsize=12)
ax.set_ylim(-2.5, 2.5)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mejor modelo (early stop): epoch {best_epoch}")
print(f"  Val loss:  {val_losses_es[best_epoch]:.4f}")
print(f"  Test loss: {test_losses_es[best_epoch]:.4f}")
print(f"\nModelo final (epoch 2000):")
print(f"  Val loss:  {val_losses_es[-1]:.4f}")
print(f"  Test loss: {test_losses_es[-1]:.4f}")
print(f"\nEarly stopping mejoró el test loss sin tocar la arquitectura ni la loss.")

El gráfico de la izquierda es **la forma clásica de detectar overfitting**: el training loss sigue bajando pero el validation loss empieza a subir. El punto donde el validation loss es mínimo es donde queremos parar.

En la práctica, early stopping se combina con otras técnicas de regularización. No son excluyentes — se complementan.

---

<a id='ensembling'></a>
## 6. Ensembling

La idea de ensembling es **entrenar múltiples modelos y promediar sus predicciones**. Suena simple, y lo es. Pero es una de las técnicas más poderosas que existen.

### ¿Por qué funciona?

Si cada modelo tiene un error con cierta componente aleatoria (ruido), al promediar muchos modelos, **los errores aleatorios se cancelan** entre sí. Solo queda la señal real.

Formalmente, si tenemos $M$ modelos con errores independientes:

$$\text{Var}\left(\frac{1}{M} \sum_{m=1}^{M} f_m(x)\right) = \frac{\text{Var}(f_m(x))}{M}$$

La varianza (el overfitting) se reduce por un factor $M$. Esto es el mismo principio que en estadística: el promedio de muchas mediciones ruidosas es más preciso que una sola medición.

**Ojo**: esto funciona **solo si los errores son independientes** (o al menos diferentes). Si todos los modelos cometen exactamente el mismo error, promediar no sirve de nada.

### Fuentes de variabilidad

Para que el ensemble funcione, necesitamos que los modelos sean **diferentes**. ¿De dónde sacamos esa diversidad?

1. **Inicialización aleatoria**: entrenar la misma arquitectura con diferentes seeds. Cada modelo encuentra un mínimo diferente.

2. **Diferentes subsets de datos (bagging)**: entrenar cada modelo con un subconjunto aleatorio del dataset (con reemplazo). Random Forest es exactamente esto.

3. **Diferentes hiperparámetros**: variar el learning rate, la arquitectura, el número de capas, etc.

4. **Diferentes checkpoints**: guardar el mismo modelo en diferentes momentos del entrenamiento (epoch 100, 200, 300...) y promediar.

### El costo del ensembling

La desventaja obvia es que entrenar $M$ modelos cuesta $M$ veces más. Y en inferencia, necesitás correr $M$ forward passes para cada predicción.

En competencias de machine learning (Kaggle), el ensembling es **obligatorio** para ganar. En producción, a veces se usa "knowledge distillation" para entrenar un solo modelo que imite al ensemble.

In [ ]:
# Demo: ensembling reduces overfitting
np.random.seed(42)

n_models = 8
models = []
individual_preds = []

x_plot_ens = np.linspace(-3.5, 3.5, 300).reshape(-1, 1)

# Train multiple models with different initializations
for seed in range(n_models):
    net = SimpleNet(hidden_size=100, seed=seed)
    for epoch in range(1500):
        net.forward(x_train_nn)
        net.backward(x_train_nn, y_train_nn, lr=0.01)
    models.append(net)
    individual_preds.append(net.forward(x_plot_ens))

# Ensemble prediction (average)
ensemble_pred = np.mean(individual_preds, axis=0)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for i, pred in enumerate(individual_preds):
    ax.plot(x_plot_ens, np.clip(pred, -3, 3), alpha=0.3, linewidth=1,
            color=plt.cm.Set2(i / n_models), label=f'Modelo {i+1}' if i < 4 else None)
ax.scatter(x_train_nn, y_train_nn, c='steelblue', s=25, zorder=5)
ax.plot(x_plot_ens, np.sin(3 * x_plot_ens), 'k--', alpha=0.5, label='f(x) real')
ax.set_title(f'{n_models} modelos individuales\n(cada uno overfittea distinto)', fontsize=11)
ax.set_ylim(-2.5, 2.5)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
# Show a single model vs ensemble
ax.plot(x_plot_ens, np.clip(individual_preds[0], -3, 3), 'r-', alpha=0.5,
        linewidth=1.5, label='Un solo modelo')
ax.plot(x_plot_ens, np.clip(ensemble_pred, -3, 3), 'b-', linewidth=2.5,
        label=f'Ensemble ({n_models} modelos)')
ax.scatter(x_train_nn, y_train_nn, c='steelblue', s=25, zorder=5)
ax.plot(x_plot_ens, np.sin(3 * x_plot_ens), 'k--', alpha=0.5, label='f(x) real')
ax.set_title('Ensemble promedia → errores se cancelan', fontsize=11)
ax.set_ylim(-2.5, 2.5)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute test errors
x_te_ens = np.linspace(-3, 3, 200).reshape(-1, 1)
y_te_ens = np.sin(3 * x_te_ens) + 0.3 * np.random.randn(200, 1)

individual_errors = []
for net in models:
    pred = net.forward(x_te_ens)
    individual_errors.append(np.mean((pred - y_te_ens) ** 2))

ensemble_test_pred = np.mean([net.forward(x_te_ens) for net in models], axis=0)
ensemble_error = np.mean((ensemble_test_pred - y_te_ens) ** 2)

print(f"Test error individual (promedio): {np.mean(individual_errors):.4f} ± {np.std(individual_errors):.4f}")
print(f"Test error del ensemble:          {ensemble_error:.4f}")
print(f"\nEl ensemble es mejor que cualquier modelo individual.")
print(f"Los errores aleatorios de cada modelo se cancelan al promediar.")

Fijate que cada modelo individual overfittea de forma diferente (porque tienen diferente inicialización aleatoria). Al promediar, las oscilaciones idiosincráticas se cancelan y queda una predicción mucho más suave y cercana a la función real.

En la práctica, ensembles de 3-5 modelos ya dan una mejora significativa. Más allá de eso, los retornos marginales decrecen.

---

<a id='dropout'></a>
## 7. Dropout

Dropout es una de las técnicas de regularización más usadas en deep learning. La idea es elegante: en cada iteración de entrenamiento, **apagamos aleatoriamente un porcentaje de las neuronas** (típicamente ~50% de las hidden units).

### ¿Cómo funciona?

En cada forward pass durante el entrenamiento:

1. Para cada neurona de una capa hidden, generamos un número aleatorio entre 0 y 1
2. Si el número es menor que $p$ (la probabilidad de dropout), la neurona se "apaga": su output se pone en 0
3. Las neuronas que sobreviven se escalan por $\frac{1}{1-p}$ para mantener la escala

Matemáticamente, para una capa con activaciones $h$:

$$\text{mask} \sim \text{Bernoulli}(1 - p)$$
$$\tilde{h} = \frac{\text{mask} \odot h}{1 - p}$$

Donde $\odot$ es multiplicación elemento a elemento.

### ¿Por qué regulariza?

**1. No puede depender de pocas neuronas**

Sin dropout, la red podría aprender a depender fuertemente de unas pocas neuronas "estrella" que memorizan patrones específicos del training set. Con dropout, cualquier neurona puede desaparecer en cualquier momento, así que la red **tiene que distribuir la información** entre muchas neuronas. Es como si le dijeras: "no confíes en ninguna neurona individual, porque cualquiera puede faltar".

**2. Es como un ensemble implícito**

Cada vez que aplicamos dropout, estamos entrenando una **sub-red diferente** (la red original menos las neuronas apagadas). Con $n$ neuronas, hay $2^n$ posibles sub-redes. Dropout las entrena todas simultáneamente (con pesos compartidos). En inferencia, usar todas las neuronas (con el scaling) es equivalente a promediar las predicciones de todas las sub-redes.

Es un ensemble de $2^n$ modelos por el costo de entrenar uno solo.

**3. Rompe las co-adaptaciones**

Sin dropout, pares de neuronas pueden "co-adaptarse": la neurona A aprende un feature raro y la neurona B aprende a compensarlo. Juntas memorizan algo, pero individualmente no sirven. Dropout rompe estas co-adaptaciones porque no puede garantizar que A y B estén activas al mismo tiempo.

**4. Rompe el flujo de gradientes para esas rutas**

Cuando una neurona está apagada, no propaga gradientes hacia atrás por esa ruta. Esto es como podar aleatoriamente el grafo computacional en cada paso. El gradiente tiene que encontrar **múltiples caminos** para llegar a las capas tempranas, en vez de depender de un solo camino dominante.

### Inferencia: el scaling

En entrenamiento, cada neurona está activa con probabilidad $(1-p)$. Entonces el valor esperado de la output de la capa es $(1-p) \cdot h$. Para que la inferencia (donde **todas** las neuronas están activas) sea consistente, tenemos dos opciones:

- **Inverted dropout** (el estándar): durante entrenamiento, escalar las neuronas activas por $\frac{1}{1-p}$. En inferencia, no hacer nada.
- **Standard dropout**: durante entrenamiento, no escalar. En inferencia, multiplicar todos los weights por $(1-p)$.

Inverted dropout es más práctico porque no necesitás cambiar nada en inferencia.

In [ ]:
# Dropout from scratch + visual demo
np.random.seed(42)

class SimpleNetDropout:
    """2-layer neural network with dropout."""
    def __init__(self, hidden_size, seed=42):
        np.random.seed(seed)
        self.W1 = np.random.randn(1, hidden_size) * np.sqrt(2.0 / 1)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, 1) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, 1))
        self.hidden_size = hidden_size
    
    def forward(self, x, dropout_prob=0.0, training=True):
        self.z1 = x @ self.W1 + self.b1
        self.h1 = relu(self.z1)
        
        # Apply dropout (inverted)
        if training and dropout_prob > 0:
            self.mask = (np.random.rand(*self.h1.shape) > dropout_prob).astype(float)
            self.h1_dropped = self.h1 * self.mask / (1 - dropout_prob)
        else:
            self.mask = np.ones_like(self.h1)
            self.h1_dropped = self.h1
        
        self.out = self.h1_dropped @ self.W2 + self.b2
        return self.out
    
    def backward(self, x, y, lr=0.01, dropout_prob=0.0):
        n = x.shape[0]
        d_out = 2 * (self.out - y) / n
        # Layer 2 grads
        dW2 = self.h1_dropped.T @ d_out
        db2 = d_out.sum(axis=0, keepdims=True)
        # Layer 1 grads (through dropout mask)
        d_h1 = d_out @ self.W2.T
        d_h1 = d_h1 * self.mask / (1 - dropout_prob) if dropout_prob > 0 else d_h1
        d_z1 = d_h1 * relu_grad(self.z1)
        dW1 = x.T @ d_z1
        db1 = d_z1.sum(axis=0, keepdims=True)
        # Update
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
    
    def loss(self, x, y):
        pred = self.forward(x, dropout_prob=0.0, training=False)
        return np.mean((pred - y) ** 2)

# Visualize what dropout looks like
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

hidden_size_demo = 20
activation = np.random.rand(1, hidden_size_demo) * 2  # fake activations

for ax, dp, title in zip(axes, [0.0, 0.3, 0.5],
                          ['Sin dropout', 'Dropout p=0.3', 'Dropout p=0.5']):
    if dp > 0:
        mask = (np.random.rand(1, hidden_size_demo) > dp).astype(float)
        dropped = activation * mask / (1 - dp)
    else:
        mask = np.ones((1, hidden_size_demo))
        dropped = activation
    
    colors = ['green' if m == 1 else 'red' for m in mask[0]]
    ax.bar(range(hidden_size_demo), dropped[0], color=colors, alpha=0.7,
           edgecolor='black', linewidth=0.5)
    ax.bar(range(hidden_size_demo), activation[0], color='lightblue', alpha=0.3,
           edgecolor='gray', linewidth=0.3, label='Original')
    n_active = int(mask.sum())
    ax.set_title(f'{title}\n{n_active}/{hidden_size_demo} neuronas activas', fontsize=11)
    ax.set_xlabel('Neurona', fontsize=10)
    ax.set_ylabel('Activación', fontsize=10)

plt.suptitle('Dropout: neuronas verdes = activas, rojas = apagadas\n'
             '(las activas se escalan por 1/(1-p) para compensar)',
             fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Train with and without dropout
np.random.seed(42)

dropout_probs = [0.0, 0.3, 0.5]
dp_labels = ['Sin dropout', 'Dropout p=0.3', 'Dropout p=0.5']
dp_colors = ['red', 'blue', 'green']

dp_results = []

for dp in dropout_probs:
    net = SimpleNetDropout(hidden_size=200, seed=42)
    train_losses = []
    test_losses = []
    
    for epoch in range(3000):
        # Forward with dropout (training mode)
        net.forward(x_train_nn, dropout_prob=dp, training=True)
        net.backward(x_train_nn, y_train_nn, lr=0.01, dropout_prob=dp)
        
        if epoch % 10 == 0:
            # Evaluate without dropout
            train_losses.append(net.loss(x_train_nn, y_train_nn))
            test_pred = net.forward(x_te, dropout_prob=0, training=False)
            test_losses.append(np.mean((test_pred - y_te) ** 2))
    
    dp_results.append((net, train_losses, test_losses))

# Plot predictions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (net, _, _), label, color, dp in zip(axes, dp_results, dp_labels, dp_colors, dropout_probs):
    y_pred = net.forward(x_test_nn, dropout_prob=0, training=False)
    ax.scatter(x_train_nn, y_train_nn, c='steelblue', s=25, zorder=5, label='Train')
    ax.plot(x_test_nn, np.sin(3 * x_test_nn), 'k--', alpha=0.4, label='f(x) real')
    ax.plot(x_test_nn, np.clip(y_pred, -3, 3), color=color, linewidth=2, label=label)
    ax.set_title(label, fontsize=11)
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Efecto del dropout: misma red de 200 neuronas', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Plot train vs test gap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for (_, train_l, _), label, color in zip(dp_results, dp_labels, dp_colors):
    ax.plot(train_l, color=color, alpha=0.7, label=label)
ax.set_xlabel('Step (×10)', fontsize=11)
ax.set_ylabel('Train Loss', fontsize=11)
ax.set_title('Training loss', fontsize=12)
ax.legend(fontsize=9)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

ax = axes[1]
for (_, _, test_l), label, color in zip(dp_results, dp_labels, dp_colors):
    ax.plot(test_l, color=color, alpha=0.7, label=label)
ax.set_xlabel('Step (×10)', fontsize=11)
ax.set_ylabel('Test Loss', fontsize=11)
ax.set_title('Test loss (generalización)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Sin dropout:   train loss baja mucho pero test loss sube → overfitting")
print("Con dropout:   train loss baja menos pero test loss se mantiene bajo → generaliza")
print("\nDropout es GRATIS en terms of capacidad: la red sigue siendo de 200 neuronas.")
print("Solo le forzamos a no depender de pocas neuronas.")

### Tips prácticos de dropout

- **Dropout rate típico**: 0.1-0.5 para capas hidden, 0.0-0.1 para la capa de input
- **No usar en inferencia**: en inferencia, todas las neuronas están activas (y el scaling de inverted dropout compensa)
- **No usar con BatchNorm**: dropout y BatchNorm interactúan mal. En modelos modernos (Transformers), se suele usar dropout sin BatchNorm.
- **model.train() / model.eval()**: en PyTorch, **siempre** llamar `model.eval()` antes de inferencia para desactivar dropout (y BatchNorm)
- **Dropout antes o después de la activación**: ambos funcionan, pero después de la activación (después de ReLU) es lo más común

---

<a id='ruido'></a>
## 8. Agregar ruido

Una forma general de regularizar es **inyectar ruido** en diferentes partes del proceso de entrenamiento. Dropout es un caso especial (ruido multiplicativo en las activaciones), pero hay muchas otras formas:

### 8.1. Ruido en los inputs (Data Augmentation)

Es la forma más natural de inyectar ruido: modificar los datos de entrada de formas que preservan su label. En imágenes: rotar, escalar, flipear, cambiar colores, recortar, etc. Lo vemos en detalle en la sección 10.

### 8.2. Adversarial Training

Un tipo especial de ruido en los inputs: en vez de ruido aleatorio, agregamos ruido **adversarial** — la perturbación más dañina posible dentro de un radio $\epsilon$:

$$x_{adv} = x + \epsilon \cdot \text{sign}(\nabla_x L(f(x; \theta), y))$$

Esto encuentra la dirección en la que la loss **crece más rápido** respecto al input, y perturba en esa dirección. El modelo se entrena para ser robusto a estas perturbaciones, lo que mejora la generalización.

Adversarial training es más caro (necesitás calcular gradientes respecto al input, no solo respecto a los parámetros) pero produce modelos más robustos.

### 8.3. Ruido en los weights

En vez de agregar ruido a las activaciones (dropout) o a los inputs, lo podemos agregar directamente a los **pesos**:

$$\tilde{W} = W + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Esto hace que el modelo sea robusto a perturbaciones en sus propios parámetros. Es equivalente a un prior que prefiere funciones que no cambian mucho cuando los parámetros cambian un poco (funciones suaves).

### 8.4. Label Smoothing

En clasificación, en vez de usar **labels duros** (one-hot: `[0, 0, 1, 0]`), usamos **labels suaves**:

$$y_{\text{smooth}} = (1 - \alpha) \cdot y_{\text{one-hot}} + \frac{\alpha}{C}$$

Con $\alpha = 0.1$ y 4 clases, el label `[0, 0, 1, 0]` se convierte en `[0.025, 0.025, 0.925, 0.025]`.

¿Por qué? Porque el label duro le dice al modelo: "estoy 100% seguro de que es clase 3". Pero en la realidad, hay incertidumbre. Label smoothing le dice: "estoy 92.5% seguro de que es clase 3, pero hay una probabilidad chiquita de que sea otra clase".

Esto evita que el modelo produzca logits extremadamente grandes (para maximizar la confianza), lo que mejora la generalización y la calibración de las probabilidades.

In [ ]:
# Demo: Label smoothing effect on logit magnitudes
np.random.seed(42)

def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def cross_entropy(probs, targets):
    return -np.sum(targets * np.log(probs + 1e-10), axis=-1).mean()

# Show how hard labels push logits to infinity vs label smoothing
n_classes = 4
logit_range = np.linspace(0, 20, 100)

# For hard labels: loss = -log(softmax(z_true))
# As logit for true class increases, loss decreases
hard_target = np.array([0, 0, 1, 0])  # one-hot
alpha = 0.1
smooth_target = (1 - alpha) * hard_target + alpha / n_classes

losses_hard = []
losses_smooth = []

for max_logit in logit_range:
    logits = np.array([0.0, 0.0, max_logit, 0.0])
    probs = softmax(logits)
    losses_hard.append(cross_entropy(probs.reshape(1, -1), hard_target.reshape(1, -1)))
    losses_smooth.append(cross_entropy(probs.reshape(1, -1), smooth_target.reshape(1, -1)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(logit_range, losses_hard, 'r-', linewidth=2, label='Hard labels [0,0,1,0]')
ax.plot(logit_range, losses_smooth, 'b-', linewidth=2, label=f'Smooth labels {smooth_target}')
ax.set_xlabel('Logit de la clase correcta', fontsize=11)
ax.set_ylabel('Loss (cross-entropy)', fontsize=11)
ax.set_title('Hard labels empujan logits al infinito\nSmooth labels tienen un mínimo finito', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.axhline(y=-np.log(1-alpha+alpha/n_classes), color='blue', linestyle=':', alpha=0.5)

# Show the different targets visually
ax = axes[1]
x_pos = np.arange(n_classes)
width = 0.35
ax.bar(x_pos - width/2, hard_target, width, label='Hard labels', color='red', alpha=0.7)
ax.bar(x_pos + width/2, smooth_target, width, label=f'Smooth labels (α={alpha})', color='blue', alpha=0.7)
ax.set_xlabel('Clase', fontsize=11)
ax.set_ylabel('Probabilidad target', fontsize=11)
ax.set_title('Labels duros vs suaves', fontsize=12)
ax.legend(fontsize=10)
ax.set_xticks(x_pos)
ax.set_xticklabels(['Clase 0', 'Clase 1', 'Clase 2\n(correcta)', 'Clase 3'])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Con hard labels, la loss se minimiza cuando el logit → ∞")
print("El modelo nunca está 'satisfecho', siempre quiere más confianza.")
print(f"\nCon label smoothing (α={alpha}), la loss tiene un MÍNIMO FINITO.")
print("El modelo deja de empujar los logits cuando llega a una confianza razonable.")
print("\nEsto previene overfitting a las labels exactas y mejora la calibración.")

Todas estas formas de agregar ruido comparten la misma intuición: **si el modelo funciona bien a pesar del ruido, es porque aprendió patrones robustos, no particularidades del training set.**

---

<a id='transfer-learning'></a>
## 9. Transfer learning

Transfer learning es la idea de **tomar un modelo pre-entrenado en una tarea grande y reutilizarlo para una tarea nueva más chica**. Es probablemente la técnica más impactante de deep learning moderno.

### ¿Por qué funciona?

Las primeras capas de una red neuronal (especialmente en visión) aprenden features **generales**:
- Capa 1: bordes, texturas básicas
- Capa 2: formas simples, patrones
- Capa 3: partes de objetos
- Última capa: conceptos específicos de la tarea ("es un gato", "es un perro")

Los features de las primeras capas son **universales**: sirven para cualquier tarea de visión. Solo la última capa es específica. Entonces, ¿para qué aprender todo de cero si las primeras capas van a aprender lo mismo?

### El procedimiento

1. **Tomar un modelo pre-entrenado**: por ejemplo, un ResNet-50 entrenado en ImageNet (1.3 millones de imágenes, 1000 clases)

2. **Quitar la última capa** (el "head" de clasificación): reemplazarla por una nueva capa con el número de clases de tu tarea

3. **Fine-tuning**: entrenar con tus datos, típicamente con:
   - **LR bajo** para las capas pre-entrenadas (no querés destruir lo que ya aprendieron)
   - **LR normal** para la nueva última capa (necesita aprender de cero)
   
Opcionalmente, podés **congelar** las primeras capas (no actualizar sus parámetros) y solo entrenar las últimas.

```python
# PyTorch: transfer learning en 5 líneas
import torchvision.models as models

# 1. Cargar modelo pre-entrenado
model = models.resnet50(pretrained=True)

# 2. Congelar todas las capas
for param in model.parameters():
    param.requires_grad = False

# 3. Reemplazar la última capa (nueva tarea: 5 clases)
model.fc = nn.Linear(model.fc.in_features, 5)
# Solo model.fc tiene requires_grad=True

# 4. Entrenar solo la última capa
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)
```

### ¿Cuánto fine-tuning hacer?

Depende de cuántos datos tenés para la nueva tarea:

| Datos nuevos | Estrategia |
|:-------------|:-----------|
| Muy pocos (<100) | Congelar todo, solo entrenar la última capa |
| Pocos (100-1000) | Congelar primeras capas, fine-tune últimas capas |
| Muchos (>10000) | Fine-tune todo el modelo con LR bajo |
| Muchísimos | Tal vez convenga entrenar desde cero |

### ¿Por qué es regularización?

Transfer learning actúa como regularización porque:

1. **Prior informativo**: en vez de empezar de cero (prior uniforme), empezás de parámetros que ya capturan features útiles. Es como un prior MUY fuerte.

2. **Menos parámetros que ajustar**: si congelás capas, reducís drásticamente los parámetros libres, lo que reduce overfitting.

3. **Features estables**: los features pre-entrenados son estables y generales, no se ajustan al ruido de tu dataset chiquito.

Hoy en día, transfer learning es el **estándar**. Casi nadie entrena modelos grandes desde cero. Se toma un foundation model (GPT, BERT, CLIP, etc.) y se fine-tunea para la tarea específica.

In [ ]:
# Conceptual demo: transfer learning with a simple model
np.random.seed(42)

# Simulate a "pre-trained" feature extractor
# Trained on a "large dataset" to learn good features
def pretrained_features(x):
    """Simulate a pre-trained model: extracts sin and cos features.
    In a real scenario, this would be the output of a ResNet/BERT/etc."""
    return np.column_stack([np.sin(x), np.cos(x), np.sin(2*x), np.cos(2*x)])

# New task: very few training examples
n_few = 10
x_few = np.sort(np.random.uniform(-3, 3, n_few)).reshape(-1, 1)
y_few = (0.5 * np.sin(2 * x_few) + 0.3 * np.cos(x_few) +
         0.1 * np.random.randn(n_few, 1))

x_plot_tl = np.linspace(-3.5, 3.5, 300).reshape(-1, 1)
y_true_tl = 0.5 * np.sin(2 * x_plot_tl) + 0.3 * np.cos(x_plot_tl)

# Approach 1: Train from scratch (overparameterized network)
net_scratch = SimpleNet(hidden_size=100, seed=42)
for epoch in range(2000):
    net_scratch.forward(x_few)
    net_scratch.backward(x_few, y_few, lr=0.005)
y_scratch = net_scratch.forward(x_plot_tl)

# Approach 2: Transfer learning (use pre-trained features + simple linear head)
features_train = pretrained_features(x_few)
features_plot = pretrained_features(x_plot_tl)

# Just a linear regression on the pre-trained features (very few parameters!)
# Using closed-form solution: w = (X^T X)^{-1} X^T y
X_aug = np.column_stack([features_train, np.ones((n_few, 1))])  # add bias
w_transfer = np.linalg.lstsq(X_aug, y_few, rcond=None)[0]
X_plot_aug = np.column_stack([features_plot, np.ones((len(x_plot_tl), 1))])
y_transfer = X_plot_aug @ w_transfer

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(x_few, y_few, c='steelblue', s=60, zorder=5, label=f'Train ({n_few} datos)')
ax.plot(x_plot_tl, y_true_tl, 'k--', alpha=0.5, label='f(x) real')
ax.plot(x_plot_tl, np.clip(y_scratch, -3, 3), 'r-', linewidth=2,
        label='Desde cero (100 neuronas)')
ax.set_title('Entrenado desde cero\n(muchos parámetros, pocos datos → overfitting)', fontsize=11)
ax.set_ylim(-2, 2)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(x_few, y_few, c='steelblue', s=60, zorder=5, label=f'Train ({n_few} datos)')
ax.plot(x_plot_tl, y_true_tl, 'k--', alpha=0.5, label='f(x) real')
ax.plot(x_plot_tl, y_transfer, 'g-', linewidth=2,
        label='Transfer learning (features pre-entrenados)')
ax.set_title('Transfer learning\n(features pre-entrenados + linear head → generaliza)', fontsize=11)
ax.set_ylim(-2, 2)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Solo {n_few} datos de entrenamiento', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Modelo desde cero: 100×2 + 100 + 1×100 + 1 = ~400 parámetros para {n_few} datos")
print(f"Transfer learning: 4+1 = 5 parámetros (solo el linear head) sobre features pre-entrenados")
print(f"\nCuando tenés pocos datos, transfer learning es MUCHO más efectivo.")
print(f"Los features pre-entrenados actúan como un prior muy fuerte.")

---

<a id='data-augmentation'></a>
## 10. Data augmentation

Data augmentation es quizás la técnica de regularización más intuitiva: **crear más datos de entrenamiento a partir de los existentes**, aplicando transformaciones que preservan el label.

### La intuición

Si tenés una foto de un gato y la rotás 15 grados, sigue siendo un gato. Si la espejás horizontalmente, sigue siendo un gato. Si le cambiás un poco el brillo, sigue siendo un gato.

Cada transformación genera un "dato nuevo" que el modelo tiene que clasificar correctamente. Esto:

1. **Aumenta el dataset efectivo**: más datos = menos overfitting
2. **Enseña invarianzas**: el modelo aprende que la identidad del objeto no cambia con rotación, escala, etc.
3. **Es gratis**: no necesitás etiquetar datos nuevos

### Transformaciones comunes en imágenes

| Transformación | Descripción |
|:---------------|:------------|
| **Random crop** | Recortar una región aleatoria |
| **Horizontal flip** | Espejo horizontal (NO vertical — un gato boca abajo es raro) |
| **Rotation** | Rotar ±15-30 grados |
| **Color jitter** | Cambiar brillo, contraste, saturación |
| **Random erasing** | Borrar un rectángulo aleatorio (enseña al modelo a no depender de una sola región) |
| **Mixup** | Mezclar dos imágenes y sus labels: $x' = \lambda x_1 + (1-\lambda) x_2$, $y' = \lambda y_1 + (1-\lambda) y_2$ |
| **CutMix** | Pegar una región de una imagen sobre otra |

### Data augmentation en otros dominios

- **Texto**: sinónimos, back-translation, eliminación aleatoria de palabras
- **Audio**: cambiar velocidad, agregar ruido de fondo, pitch shift
- **Series temporales**: time warping, window slicing, jitter
- **Tabular**: SMOTE (oversampling sintético), ruido gaussiano en features continuos

### Augmentation como regularización

Data augmentation es regularización porque **introduce un prior sobre qué transformaciones no deberían cambiar la predicción**. Es como decirle al modelo: "la rotación no importa, el flip no importa, el brillo no importa". Este prior reduce el espacio de funciones que el modelo puede aprender, favoreciendo soluciones invariantes a estas transformaciones.

In [ ]:
# Demo: data augmentation effect on 1D regression
np.random.seed(42)

# Few training points
n_orig = 15
x_orig = np.sort(np.random.uniform(-2.5, 2.5, n_orig)).reshape(-1, 1)
y_orig = np.sin(2 * x_orig) + 0.2 * np.random.randn(n_orig, 1)

# Augment: add noisy copies of the data
def augment_data(x, y, n_augmented=5, noise_x=0.1, noise_y=0.05):
    """Create augmented dataset by adding noise to inputs and outputs."""
    x_aug_list = [x]
    y_aug_list = [y]
    for _ in range(n_augmented):
        x_noisy = x + noise_x * np.random.randn(*x.shape)
        y_noisy = y + noise_y * np.random.randn(*y.shape)
        x_aug_list.append(x_noisy)
        y_aug_list.append(y_noisy)
    return np.vstack(x_aug_list), np.vstack(y_aug_list)

x_aug, y_aug = augment_data(x_orig, y_orig, n_augmented=10)

# Train without augmentation
net_no_aug = SimpleNet(hidden_size=100, seed=42)
for epoch in range(2000):
    net_no_aug.forward(x_orig)
    net_no_aug.backward(x_orig, y_orig, lr=0.01)

# Train with augmentation
net_aug = SimpleNet(hidden_size=100, seed=42)
for epoch in range(2000):
    # Re-augment each epoch (different noise each time)
    x_a, y_a = augment_data(x_orig, y_orig, n_augmented=5, noise_x=0.15, noise_y=0.05)
    net_aug.forward(x_a)
    net_aug.backward(x_a, y_a, lr=0.01)

# Plot
x_plot_da = np.linspace(-3, 3, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original data
ax = axes[0]
ax.scatter(x_orig, y_orig, c='steelblue', s=40, zorder=5, label='Datos originales')
ax.scatter(x_aug, y_aug, c='orange', s=8, alpha=0.3, label='Datos augmentados')
ax.plot(x_plot_da, np.sin(2 * x_plot_da), 'k--', alpha=0.4, label='f(x) real')
ax.set_title(f'Datos: {n_orig} originales → {len(x_aug)} con augmentation', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(-2.5, 2.5)
ax.grid(True, alpha=0.3)

# Without augmentation
ax = axes[1]
y_pred_no = net_no_aug.forward(x_plot_da)
ax.scatter(x_orig, y_orig, c='steelblue', s=40, zorder=5)
ax.plot(x_plot_da, np.sin(2 * x_plot_da), 'k--', alpha=0.4, label='f(x) real')
ax.plot(x_plot_da, np.clip(y_pred_no, -3, 3), 'r-', linewidth=2, label='Sin augmentation')
ax.set_title('Sin augmentation → overfitting', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(-2.5, 2.5)
ax.grid(True, alpha=0.3)

# With augmentation
ax = axes[2]
y_pred_aug = net_aug.forward(x_plot_da)
ax.scatter(x_orig, y_orig, c='steelblue', s=40, zorder=5)
ax.plot(x_plot_da, np.sin(2 * x_plot_da), 'k--', alpha=0.4, label='f(x) real')
ax.plot(x_plot_da, np.clip(y_pred_aug, -3, 3), 'g-', linewidth=2, label='Con augmentation')
ax.set_title('Con augmentation → generaliza mejor', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(-2.5, 2.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Data augmentation funciona porque:")
print("1. Más datos → menos overfitting (la proporción parámetros/datos mejora)")
print("2. El ruido en los inputs actúa como regularización implícita")
print("3. El modelo aprende a ser robusto a pequeñas perturbaciones")
print("\nEn imágenes, las augmentaciones (flip, crop, color jitter) son ESENCIALES.")
print("Sin augmentation, hasta los modelos más grandes overfittean.")

Data augmentation es tan efectiva que es considerada **parte estándar** del pipeline de entrenamiento, no un "truco" opcional. En visión por computadora, un modelo entrenado sin augmentation no tiene chances de competir con uno que la usa.

---

<a id='resumen'></a>
## 11. Resumen

Todas las técnicas de regularización comparten un objetivo: **reducir la varianza del modelo (overfitting) sin aumentar demasiado el bias (underfitting)**. Lo hacen de formas diferentes pero complementarias.

| Técnica | Tipo | Descripción | Cuándo usar |
|:--------|:-----|:------------|:------------|
| **L2 / weight decay** | Explícita | Penaliza pesos grandes → función suave | Siempre (viene por default en AdamW) |
| **L1** | Explícita | Penaliza pesos no-cero → sparsity | Cuando querés selección de features |
| **SGD con mini-batches** | Implícita | Ruido del gradiente favorece mínimos anchos | Siempre (nadie usa full-batch) |
| **Early stopping** | Implícita | Parar antes de overfittear | Siempre (primera línea de defensa) |
| **Ensembling** | Modelo | Promediar múltiples modelos | Cuando necesitás el máximo rendimiento |
| **Dropout** | Explícita | Apagar neuronas aleatorias | Redes medianas/grandes sin BatchNorm |
| **Label smoothing** | Ruido | Suavizar labels → logits más moderados | Clasificación con muchas clases |
| **Data augmentation** | Datos | Más datos con transformaciones | Siempre en visión, a menudo en NLP |
| **Transfer learning** | Prior | Empezar de modelo pre-entrenado | Cuando tenés pocos datos para tu tarea |

### La receta moderna

En la práctica, se combinan varias técnicas:

1. **Data augmentation** (siempre, especialmente en visión)
2. **Transfer learning** (casi siempre — no entrenar desde cero si podés evitarlo)
3. **AdamW** con weight decay (el optimizer estándar ya incluye L2)
4. **Dropout** en las capas hidden (0.1-0.5)
5. **Early stopping** monitoreando el validation loss
6. **Label smoothing** (α=0.1 es un buen default)

Y recordá: **más datos siempre es mejor que más regularización**. Si podés conseguir más datos etiquetados, eso va a mejorar más que cualquier técnica de regularización.

---

**Siguiente notebook →** [09 - Secuencias: RNN y LSTM](./09_secuencias_rnn_lstm.ipynb): cómo hacer que las redes neuronales procesen datos secuenciales (texto, audio, series de tiempo).